# w6 — GRASP planner: benchmark vs CEM & vanilla GD on Push-T

Implements **GRASP** (Gradient-based Randomized Adaptive Search Planner)
on top of a trained **LeWM**, and benchmarks it against **CEM** and
**vanilla gradient descent** — measuring **success rate** and **wall-clock
planning time** in the real Push-T env.

Needs a trained LeWM checkpoint. Steps 1–3 reproduce the w4 setup so this
notebook is self-contained; if you already trained LeWM in this session,
skip to step 4. Runtime → **T4 GPU**.

## 0. GPU

In [ ]:
!nvidia-smi -L
import torch; assert torch.cuda.is_available(), 'Enable a GPU runtime!'

## 1. Install + clone (same as w4)

In [ ]:
%%bash
set -e
pip -q install 'stable-worldmodel[train,env]' zstandard imageio 2>&1 | tail -1
apt-get -qq install -y xvfb zstd > /dev/null
cd /content && [ -d le-wm ] || git clone -q https://github.com/lucas-maes/le-wm.git
echo done

In [ ]:
import os, sys
os.environ['STABLEWM_HOME'] = '/content/.stable-wm'
os.makedirs(os.environ['STABLEWM_HOME'], exist_ok=True)
os.chdir('/content/le-wm'); sys.path.insert(0, '/content/le-wm')

## 2. Data (HDF5 from HF → Lance) — skip if already present

In [ ]:
import os
home = os.environ['STABLEWM_HOME']
h5 = os.path.join(home, 'pusht_expert_train.h5')
lance = os.path.join(home, 'pusht_expert_train.lance')
if not os.path.exists(lance):
    from huggingface_hub import hf_hub_download
    z = hf_hub_download('quentinll/lewm-pusht', 'pusht_expert_train.h5.zst',
                        repo_type='dataset', local_dir=home)
    os.system(f'zstd -d -f {z} -o {h5} && rm -f {z}')
    from stable_worldmodel.data import convert
    convert(h5, lance, dest_format='lance', mode='overwrite')
print('data ready')

## 3. Train LeWM from scratch — skip if checkpoint exists

Same command as w4. ~1.5–3 h on a T4. If you saved a checkpoint from w4,
upload it to `$STABLEWM_HOME/checkpoints/pusht/lewm/` and skip this.

In [ ]:
%%bash
cd /content/le-wm; export STABLEWM_HOME=/content/.stable-wm
CKPT=$STABLEWM_HOME/checkpoints/pusht/lewm/weights_epoch_20.pt
if [ -f "$CKPT" ]; then echo "checkpoint exists, skipping training"; else \
python train.py data=pusht \
  img_size=112 trainer.max_epochs=20 trainer.precision=16-mixed \
  trainer.accelerator=gpu trainer.devices=1 loader.batch_size=96 \
  num_workers=2 output_model_name=pusht/lewm ; fi

## 4. Add the GRASP planner + benchmark scripts

In [ ]:
%%writefile /content/le-wm/grasp_solver.py
"""GRASP — Gradient-based Randomized Adaptive Search Planner.

A model-based planner for LeWM (or any `Costable` world model exposing
``get_cost(info_dict, action_candidates) -> (batch, num_samples)`` that is
differentiable w.r.t. the actions).

Motivation
----------
The two standard planners have complementary weaknesses on the rugged cost
landscape induced by an autoregressive world-model rollout:

* **CEM** (sampling): global but gradient-free — needs many samples/iterations
  to refine precise actions, so it is slow (wall-clock heavy) and its elite
  distribution can collapse prematurely.
* **Vanilla gradient descent**: fast local refinement, but a single (or few)
  start gets trapped in local minima, is sensitive to initialization, and can
  diverge (exploding gradients through a long rollout).

GRASP is the classic *Greedy Randomized Adaptive Search Procedure* specialized
to trajectory optimization: it interleaves the two.

Algorithm (per replan, per env, all vectorized)
------------------------------------------------
1. **Randomized adaptive construction** — sample a population of candidate
   action sequences around the (warm-started) mean; keep the incumbent as
   sample 0 (greedy elitism).
2. **Local search** — refine the whole population in parallel with `inner_steps`
   of Adam through the differentiable cost, with gradient clipping + action
   clamping for stability.
3. **Adaptive restart with elite memory** — refit a Gaussian to the top-`elite`
   candidates (CEM-style), keep the best-so-far, and resample the population for
   the next round. Repeat for `rounds`.
4. Return the single lowest-cost action sequence found.

This drops into `stable_worldmodel` exactly like `CEMSolver` / `GradientSolver`
(same `model=` constructor, `configure()`, and `solve()` contract), so it can be
used by `WorldModelPolicy` and le-wm's `eval.py` with no other changes.
"""

import time
from typing import Any

import numpy as np
import torch

try:  # use the library's warm-start helper when available
    from stable_worldmodel.solver.utils import prepare_init_action
except Exception:  # offline / unit-test fallback (matches the swm semantics)
    def prepare_init_action(model, info_dict, init_action, horizon, n_envs, action_dim):
        n_prev = init_action.shape[1] if init_action is not None else 0
        remaining = horizon - n_prev
        if remaining <= 0:
            return init_action
        device = init_action.device if init_action is not None else "cpu"
        tail = torch.zeros(n_envs, remaining, action_dim, device=device)
        return tail if init_action is None else torch.cat([init_action, tail], dim=1)


class GRASPSolver:
    """Gradient-based Randomized Adaptive Search Planner.

    Args:
        model: world model implementing the Costable protocol (``get_cost``).
        num_restarts: population size per env (parallel candidate sequences).
        rounds: number of adaptive-restart rounds (outer loop).
        inner_steps: gradient (Adam) steps of local search per round.
        elite: number of elites kept to refit the sampling distribution.
        lr: Adam learning rate for the local search.
        var_scale: initial std of the construction sampling.
        var_floor: minimum elite std (prevents premature collapse).
        grad_clip: max grad-norm for the action tensor (stability).
        action_noise: optional exploration noise added after each Adam step.
        batch_size: envs processed at once (None = all).
        device, seed: as usual.
    """

    def __init__(
        self,
        model,
        num_restarts: int = 32,
        rounds: int = 3,
        inner_steps: int = 10,
        elite: int = 8,
        lr: float = 0.1,
        var_scale: float = 1.0,
        var_floor: float = 1e-2,
        grad_clip: float | None = 1.0,
        action_noise: float = 0.0,
        batch_size: int | None = None,
        device: str | torch.device = "cpu",
        seed: int = 1234,
        callbacks: list | None = None,
    ) -> None:
        self.model = model
        self.num_restarts = num_restarts
        self.rounds = rounds
        self.inner_steps = inner_steps
        self.elite = min(elite, num_restarts)
        self.lr = lr
        self.var_scale = var_scale
        self.var_floor = var_floor
        self.grad_clip = grad_clip
        self.action_noise = action_noise
        self.batch_size = batch_size
        self.device = device
        self.torch_gen = torch.Generator(device=device).manual_seed(seed)
        self.callbacks = list(callbacks) if callbacks else []
        try:
            self._dtype = next(model.parameters()).dtype
        except (AttributeError, StopIteration):
            self._dtype = torch.float32

    # -- Solver protocol -----------------------------------------------------
    def configure(self, *, action_space, n_envs: int, config: Any) -> None:
        self._action_space = action_space
        self._n_envs = n_envs
        self._config = config
        self._action_dim = int(np.prod(action_space.shape[1:]))
        # per-dim action bounds (flattened, tiled over the action_block)
        low = np.broadcast_to(action_space.low, action_space.shape)
        high = np.broadcast_to(action_space.high, action_space.shape)
        self._low = float(np.min(low))
        self._high = float(np.max(high))
        self._configured = True

    @property
    def n_envs(self) -> int:
        return self._n_envs

    @property
    def action_dim(self) -> int:
        return self._action_dim * self._config.action_block

    @property
    def horizon(self) -> int:
        return self._config.horizon

    @property
    def dtype(self) -> torch.dtype:
        return self._dtype

    def __call__(self, *args, **kwargs) -> dict:
        return self.solve(*args, **kwargs)

    # -- helpers -------------------------------------------------------------
    def _randn(self, *shape):
        return torch.randn(*shape, generator=self.torch_gen, device=self.device, dtype=self.dtype)

    def _expand_infos(self, info_dict, start, end, pop):
        """Expand a batch slice of the info dict to (bs, pop, ...)."""
        out = {}
        for k, v in info_dict.items():
            vb = v[start:end]
            if torch.is_tensor(v):
                dt = self.dtype if vb.is_floating_point() else None
                vb = vb.to(device=self.device, dtype=dt).unsqueeze(1).expand(end - start, pop, *vb.shape[1:])
            elif isinstance(v, np.ndarray):
                vb = np.repeat(vb[:, None, ...], pop, axis=1)
            out[k] = vb
        return out

    @torch.no_grad()
    def _cost(self, infos, cands):
        return self.model.get_cost(infos, cands)

    # -- main ----------------------------------------------------------------
    def solve(self, info_dict: dict, init_action: torch.Tensor | None = None) -> dict:
        t0 = time.time()
        total_envs = len(next(iter(info_dict.values())))
        R, H, A = self.num_restarts, self.horizon, self.action_dim

        with torch.no_grad():
            init = prepare_init_action(self.model, info_dict, init_action, H, total_envs, A)
            init = init.to(self.device, self.dtype)

        bs_cfg = self.batch_size or total_envs
        best_actions, best_costs, cost_curves = [], [], []

        for s in range(0, total_envs, bs_cfg):
            e = min(s + bs_cfg, total_envs)
            bs = e - s
            infos = self._expand_infos(info_dict, s, e, R)
            batch_idx = torch.arange(bs, device=self.device)

            mean = init[s:e]                                   # (bs, H, A)
            # 1. randomized adaptive construction
            pop = mean.unsqueeze(1) + self.var_scale * self._randn(bs, R, H, A)
            pop[:, 0] = mean                                  # keep incumbent
            pop.clamp_(self._low, self._high)

            best_act = pop[:, 0].clone()
            best_cost = torch.full((bs,), float("inf"), device=self.device, dtype=self.dtype)
            curve = []

            for _ in range(self.rounds):
                # 2. local search: Adam through the differentiable cost
                leaf = pop.clone().detach().requires_grad_(True)
                opt = torch.optim.Adam([leaf], lr=self.lr)
                for _ in range(self.inner_steps):
                    costs = self.model.get_cost(infos, leaf)        # (bs, R)
                    loss = costs.sum()
                    opt.zero_grad(set_to_none=True)
                    loss.backward()
                    # stability: drop non-finite grads, clip, step, clamp
                    g = leaf.grad
                    if g is not None:
                        g.nan_to_num_(0.0, 0.0, 0.0)
                        if self.grad_clip is not None:
                            torch.nn.utils.clip_grad_norm_([leaf], self.grad_clip)
                    opt.step()
                    with torch.no_grad():
                        if self.action_noise > 0:
                            leaf += self.action_noise * self._randn(bs, R, H, A)
                        leaf.clamp_(self._low, self._high)
                    curve.append(costs.detach().min().item())
                pop = leaf.detach()

                # evaluate refined population
                costs = self._cost(infos, pop)                      # (bs, R)
                cmin, cidx = costs.min(dim=1)
                improved = cmin < best_cost
                best_cost = torch.where(improved, cmin, best_cost)
                cand_best = pop[batch_idx, cidx]
                best_act = torch.where(improved.view(bs, 1, 1), cand_best, best_act)

                # 3. adaptive restart: refit Gaussian to elites, resample
                topv, topi = torch.topk(costs, k=self.elite, dim=1, largest=False)
                elites = pop[batch_idx.unsqueeze(1), topi]          # (bs, elite, H, A)
                mean = elites.mean(dim=1)
                std = elites.std(dim=1).clamp_min(self.var_floor)
                pop = mean.unsqueeze(1) + std.unsqueeze(1) * self._randn(bs, R, H, A)
                pop[:, 0] = best_act                                # elitism
                pop.clamp_(self._low, self._high)

            best_actions.append(best_act.detach().cpu())
            best_costs.extend(best_cost.detach().cpu().tolist())
            cost_curves.append(curve)

        outputs = {
            "actions": torch.cat(best_actions, dim=0),
            "cost": best_costs,
            "cost_curve": cost_curves,
        }
        outputs["solve_time"] = time.time() - t0
        return outputs


In [ ]:
%%writefile /content/le-wm/benchmark.py
"""Benchmark GRASP vs CEM vs vanilla gradient descent on Push-T with a trained
LeWM. Runs closed-loop MPC in the real env for each planner on the SAME
start/goal pairs, and reports success rate + planning wall-clock time.

Runs on Colab / a CUDA box (needs the trained LeWM checkpoint + the Push-T env).
Adapted from le-wm/eval.py; the planners share the `swm.solver` interface, so
GRASP (grasp_solver.GRASPSolver) drops in beside the library's CEMSolver /
GradientSolver with no other changes.

Example (from le-wm/, with grasp_solver.py + this file on PYTHONPATH):
    xvfb-run -a python benchmark.py \
        --model pusht/lewm/weights_epoch_20.pt --img-size 112 --num-eval 20
"""

import argparse
import os
import time
from pathlib import Path

os.environ.setdefault("MUJOCO_GL", "egl")   # harmless for pymunk PushT

import numpy as np
import torch
from sklearn import preprocessing
from torchvision.transforms import v2 as transforms

import stable_pretraining as spt
import stable_worldmodel as swm

from grasp_solver import GRASPSolver


# --------------------------------------------------------------------------- #
class TimedSolver:
    """Wraps a solver to accumulate wall-clock time and call count of solve().
    Satisfies the runtime-checkable Solver protocol by delegation."""

    def __init__(self, solver, name):
        self.solver = solver
        self.name = name
        self.total_time = 0.0
        self.n_calls = 0
        self.n_plans = 0

    def configure(self, **kw):
        return self.solver.configure(**kw)

    @property
    def action_dim(self): return self.solver.action_dim
    @property
    def n_envs(self): return self.solver.n_envs
    @property
    def horizon(self): return self.solver.horizon

    def solve(self, info_dict, init_action=None):
        n = len(next(iter(info_dict.values())))
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        t0 = time.time()
        out = self.solver.solve(info_dict, init_action=init_action)
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        self.total_time += time.time() - t0
        self.n_calls += 1
        self.n_plans += n           # number of (env) plans produced
        return out

    __call__ = solve

    @property
    def mean_plan_ms(self):
        return 1e3 * self.total_time / max(self.n_plans, 1)


def img_transform(img_size):
    return transforms.Compose([
        transforms.ToImage(),
        transforms.ToDtype(torch.float32, scale=True),
        transforms.Normalize(**spt.data.dataset_stats.ImageNet),
        transforms.Resize(size=img_size),
    ])


def episodes_length(dataset, episodes, col):
    ep = dataset.get_col_data(col)
    step = dataset.get_col_data("step_idx")
    return np.array([np.max(step[ep == e]) + 1 for e in episodes])


def build_solver(name, model, device, seed):
    """Construct each planner with a comparable-ish budget. Wall-clock is what
    we actually report, so exact budgets need not match."""
    if name == "cem":
        return swm.solver.CEMSolver(model, num_samples=300, n_steps=30, topk=30,
                                    var_scale=1.0, device=device, seed=seed)
    if name == "gd":   # vanilla gradient descent: single start, Adam, clipped
        return swm.solver.GradientSolver(
            model, n_steps=30, num_samples=1, var_scale=1.0, device=device, seed=seed,
            optimizer_cls=torch.optim.Adam, optimizer_kwargs={"lr": 0.1}, grad_clip=1.0)
    if name == "gd_multi":  # stronger GD reference: multi-start
        return swm.solver.GradientSolver(
            model, n_steps=30, num_samples=32, var_scale=1.0, device=device, seed=seed,
            optimizer_cls=torch.optim.Adam, optimizer_kwargs={"lr": 0.1}, grad_clip=1.0)
    if name == "grasp":
        return GRASPSolver(model, num_restarts=32, rounds=3, inner_steps=10, elite=8,
                           lr=0.1, var_scale=1.0, grad_clip=1.0, device=device, seed=seed)
    raise ValueError(name)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--model", required=True, help="checkpoint under $STABLEWM_HOME/checkpoints")
    ap.add_argument("--dataset", default="pusht_expert_train")
    ap.add_argument("--img-size", type=int, default=112)
    ap.add_argument("--num-eval", type=int, default=20)
    ap.add_argument("--goal-offset", type=int, default=25)
    ap.add_argument("--eval-budget", type=int, default=50)
    ap.add_argument("--planners", default="cem,gd,grasp",
                    help="comma list: cem,gd,gd_multi,grasp")
    ap.add_argument("--seed", type=int, default=42)
    ap.add_argument("--out", default="grasp_benchmark.md")
    args = ap.parse_args()

    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"device={device}")

    # -- load model once (shared across planners) --
    model = swm.wm.utils.load_pretrained(args.model).to(device).eval()
    model.requires_grad_(False)
    model.interpolate_pos_encoding = True

    # -- dataset + fixed start/goal episodes (same for every planner) --
    dataset = swm.data.HDF5Dataset(args.dataset,
                                   keys_to_cache=["action", "proprio", "state"],
                                   cache_dir=Path(swm.data.utils.get_cache_dir()))
    col = "episode_idx" if "episode_idx" in dataset.column_names else "ep_idx"
    ep_indices = np.unique(dataset.get_col_data(col))

    process = {}
    for c in ["action", "proprio", "state"]:
        if c not in dataset.column_names:
            continue
        sc = preprocessing.StandardScaler()
        d = dataset.get_col_data(c)
        sc.fit(d[~np.isnan(d).any(axis=1)])
        process[c] = sc
        if c != "action":
            process[f"goal_{c}"] = sc

    ep_len = episodes_length(dataset, ep_indices, col)
    max_start = {e: ep_len[i] - args.goal_offset - 1 for i, e in enumerate(ep_indices)}
    per_row = np.array([max_start[e] for e in dataset.get_col_data(col)])
    valid = np.nonzero(dataset.get_col_data("step_idx") <= per_row)[0]
    rng = np.random.default_rng(args.seed)
    pick = np.sort(valid[rng.choice(len(valid) - 1, size=args.num_eval, replace=False)])
    eval_eps = dataset.get_row_data(pick)[col].tolist()
    eval_start = dataset.get_row_data(pick)["step_idx"].tolist()

    tf = {"pixels": img_transform(args.img_size), "goal": img_transform(args.img_size)}
    plan_cfg = swm.PlanConfig(horizon=5, receding_horizon=5, action_block=5)

    callables = [
        {"method": "_set_state", "args": {"state": {"value": "state"}}},
        {"method": "_set_goal_state", "args": {"goal_state": {"value": "goal_state"}}},
    ]

    rows = []
    for name in args.planners.split(","):
        name = name.strip()
        print(f"\n===== planner: {name} =====")
        world = swm.World(env_name="swm/PushT-v1", num_envs=args.num_eval,
                          max_episode_steps=2 * args.eval_budget, image_shape=(224, 224))
        solver = TimedSolver(build_solver(name, model, device, args.seed), name)
        policy = swm.policy.WorldModelPolicy(solver=solver, config=plan_cfg,
                                             process=process, transform=tf)
        world.set_policy(policy)

        t0 = time.time()
        metrics = world.evaluate(
            dataset=dataset, start_steps=eval_start, goal_offset=args.goal_offset,
            eval_budget=args.eval_budget, episodes_idx=eval_eps, callables=callables)
        wall = time.time() - t0

        sr = metrics.get("success_rate", metrics.get("success", float("nan")))
        rows.append((name, sr, solver.mean_plan_ms, solver.n_calls, wall))
        print(f"  success={sr}  mean_plan={solver.mean_plan_ms:.1f} ms  total={wall:.1f}s")

    # -- comparison table --
    print(f"\n{'planner':<10}{'success':>10}{'plan(ms)':>12}{'#plans':>9}{'total(s)':>10}")
    print("-" * 51)
    for n, sr, ms, nc, wall in rows:
        srv = f"{sr:.3f}" if isinstance(sr, (int, float)) else str(sr)
        print(f"{n:<10}{srv:>10}{ms:>12.1f}{nc:>9}{wall:>10.1f}")

    with open(args.out, "w") as f:
        f.write(f"# GRASP vs CEM vs GD on Push-T (LeWM `{args.model}`, "
                f"{args.num_eval} episodes, img={args.img_size})\n\n")
        f.write("| planner | success rate | mean plan time (ms) | # replans | total wall (s) |\n")
        f.write("|---|---:|---:|---:|---:|\n")
        for n, sr, ms, nc, wall in rows:
            srv = f"{sr:.3f}" if isinstance(sr, (int, float)) else str(sr)
            f.write(f"| {n} | {srv} | {ms:.1f} | {nc} | {wall:.1f} |\n")
    print(f"\nwrote {args.out}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /content/le-wm/synthetic_benchmark.py
"""Offline benchmark of CEM vs vanilla GD vs GRASP on a rugged cost.

We can't run the real Push-T env locally (needs the trained LeWM + CUDA), so we
stress the *optimizers* on a controllable differentiable landscape that mimics
what makes world-model planning hard: a Rastrigin-style cost, which is smooth
and differentiable (GD can flow) but riddled with local minima (single-start GD
gets trapped). Each "env" is an independent planning problem; we report solution
quality, success rate, wall-clock, and stability across seeds.

The reference CEM/GD here mirror `stable_worldmodel`'s algorithms so the
comparison is apples-to-apples; on Colab, `benchmark.py` uses the library's own
`CEMSolver`/`GradientSolver` with the real LeWM. GRASP is the exact same
`grasp_solver.GRASPSolver` used everywhere.

Run:  python synthetic_benchmark.py
"""

import math
import time
import types

import numpy as np
import torch
import torch.nn as nn

from grasp_solver import GRASPSolver

torch.manual_seed(0)
DEVICE = "cpu"
DT = torch.float32


# --------------------------------------------------------------------------- #
#  Synthetic differentiable world-model cost (Rastrigin over the trajectory)   #
# --------------------------------------------------------------------------- #
class RastriginCostModel(nn.Module):
    """get_cost(info, cands) -> (bs, S). Global min where the whole action
    sequence equals a per-env target; many local minima around it.

    `amp` controls ruggedness: a dominant quadratic basin (gradients informative)
    with sinusoidal ripples (local minima that trap naive gradient descent) — a
    fair proxy for a world-model cost, unlike a maximally-adversarial landscape.
    Counts cost evaluations for budget-matched comparison."""

    def __init__(self, targets, amp=2.5, freq=0.7):
        super().__init__()
        self.register_buffer("targets", targets)          # (n_envs, H, A)
        self.amp, self.freq = amp, freq
        self._p = nn.Parameter(torch.zeros(1))            # for dtype detection
        self.n_evals = 0

    def get_cost(self, info, cands):
        # cands: (bs, S, H, A); targets sliced to this batch live in info["_tgt"]
        self.n_evals += cands.shape[0] * cands.shape[1]   # count candidate evals
        tgt = info["_tgt"]                                # (bs, S, H, A)
        d = cands - tgt
        quad = d.pow(2)
        ripple = self.amp * (1 - torch.cos(2 * math.pi * self.freq * d))
        return (quad + ripple).mean(dim=(2, 3))          # (bs, S)


def make_infos(targets, n_envs):
    # info dict whose leading dim is n_envs (so solvers read total_envs); the
    # per-env target is carried in "_tgt" and expanded by the solver.
    return {"_tgt": targets, "state": torch.zeros(n_envs, 1)}


# --------------------------------------------------------------------------- #
#  Reference CEM / GD (mirror stable_worldmodel; model= interface)            #
# --------------------------------------------------------------------------- #
class _Base:
    def configure(self, *, action_space, n_envs, config):
        self._as, self._n, self._cfg = action_space, n_envs, config
        self._ad = int(np.prod(action_space.shape[1:]))
        self.low, self.high = float(action_space.low.min()), float(action_space.high.max())

    @property
    def horizon(self): return self._cfg.horizon
    @property
    def action_dim(self): return self._ad * self._cfg.action_block
    def __call__(self, *a, **k): return self.solve(*a, **k)

    def _expand(self, info, S):
        out = {}
        for k, v in info.items():
            out[k] = v.unsqueeze(1).expand(v.shape[0], S, *v.shape[1:]) if torch.is_tensor(v) else v
        return out


class RefCEM(_Base):
    def __init__(self, model, num_samples=300, n_steps=30, topk=30, var_scale=1.0, seed=0, **_):
        self.model, self.num_samples, self.n_steps = model, num_samples, n_steps
        self.topk, self.var_scale = topk, var_scale
        self.gen = torch.Generator().manual_seed(seed)

    @torch.no_grad()
    def solve(self, info, init_action=None):
        t0 = time.time()
        E = next(iter(info.values())).shape[0]
        H, A, S = self.horizon, self.action_dim, self.num_samples
        mean = torch.zeros(E, H, A); var = self.var_scale * torch.ones(E, H, A)
        infos = self._expand(info, S)
        bidx = torch.arange(E)
        for _ in range(self.n_steps):
            c = torch.randn(E, S, H, A, generator=self.gen) * var.unsqueeze(1) + mean.unsqueeze(1)
            c[:, 0] = mean
            c.clamp_(self.low, self.high)
            costs = self.model.get_cost(infos, c)
            tv, ti = torch.topk(costs, self.topk, dim=1, largest=False)
            elites = c[bidx.unsqueeze(1), ti]
            mean, var = elites.mean(1), elites.std(1).clamp_min(1e-3)
        costs = self.model.get_cost(infos, mean.unsqueeze(1))[:, 0]
        return {"actions": mean, "cost": costs.tolist(), "solve_time": time.time() - t0}


class RefGD(_Base):
    def __init__(self, model, n_steps=30, num_samples=1, lr=0.1, var_scale=1.0,
                 grad_clip=None, seed=0, **_):
        self.model, self.n_steps, self.num_samples = model, n_steps, num_samples
        self.lr, self.var_scale, self.grad_clip = lr, var_scale, grad_clip
        self.gen = torch.Generator().manual_seed(seed)

    def solve(self, info, init_action=None):
        t0 = time.time()
        E = next(iter(info.values())).shape[0]
        H, A, S = self.horizon, self.action_dim, self.num_samples
        infos = self._expand(info, S)
        pop = torch.randn(E, S, H, A, generator=self.gen) * self.var_scale
        pop.clamp_(self.low, self.high)
        leaf = pop.clone().requires_grad_(True)
        opt = torch.optim.Adam([leaf], lr=self.lr)
        diverged = False
        for _ in range(self.n_steps):
            costs = self.model.get_cost(infos, leaf)
            opt.zero_grad(set_to_none=True)
            costs.sum().backward()
            if self.grad_clip is not None:
                torch.nn.utils.clip_grad_norm_([leaf], self.grad_clip)
            opt.step()
            with torch.no_grad():
                leaf.clamp_(self.low, self.high)
        with torch.no_grad():
            costs = self.model.get_cost(infos, leaf)
            cmin, cidx = costs.min(dim=1)
            best = leaf[torch.arange(E), cidx]
            if not torch.isfinite(cmin).all():
                diverged = True
        return {"actions": best, "cost": cmin.tolist(), "solve_time": time.time() - t0,
                "diverged": diverged}


class RefGD_SGD(_Base):
    """Deliberately unstable GD: plain SGD, high lr, NO clip / NO clamp — shows
    the exploding-gradient failure mode (actions blow up to non-finite cost)."""

    def __init__(self, model, n_steps=30, num_samples=1, lr=3.0, seed=0, **_):
        self.model, self.n_steps, self.num_samples, self.lr = model, n_steps, num_samples, lr
        self.gen = torch.Generator().manual_seed(seed)

    def solve(self, info, init_action=None):
        t0 = time.time()
        E = next(iter(info.values())).shape[0]
        H, A, S = self.horizon, self.action_dim, self.num_samples
        infos = self._expand(info, S)
        leaf = (torch.randn(E, S, H, A, generator=self.gen)).requires_grad_(True)
        opt = torch.optim.SGD([leaf], lr=self.lr)
        for _ in range(self.n_steps):
            costs = self.model.get_cost(infos, leaf)
            opt.zero_grad(set_to_none=True)
            torch.nan_to_num(costs, 1e9, 1e9, -1e9).sum().backward()
            opt.step()                                    # no clip, no clamp
        with torch.no_grad():
            costs = self.model.get_cost(infos, leaf)
            cmin, cidx = torch.nan_to_num(costs, float("inf")).min(dim=1)
            best = leaf[torch.arange(E), cidx]
        return {"actions": best, "cost": cmin.tolist(), "solve_time": time.time() - t0,
                "diverged": bool((~torch.isfinite(cmin)).any())}


# --------------------------------------------------------------------------- #
def run(planner, model, info):
    model.n_evals = 0
    out = planner.solve(info)
    costs = torch.tensor(out["cost"])
    return costs, out["solve_time"], model.n_evals, out.get("diverged", False)


def main():
    N_ENVS, H, A = 64, 5, 2          # 64 independent planning problems
    SUCCESS_THRESH = 0.5             # cost below this ~ found the global basin
    SEEDS = [0, 1, 2]

    cfg = types.SimpleNamespace(horizon=H, action_block=1, receding_horizon=H)
    bound = 4.0
    action_space = types.SimpleNamespace(
        shape=(N_ENVS, A), low=np.array(-bound, dtype=np.float32),
        high=np.array(bound, dtype=np.float32))

    # ~1000-eval budget shared by the budget-matched planners
    planners = [
        ("CEM (big, 9000 evals)", lambda m, s: RefCEM(m, num_samples=300, n_steps=30, topk=30, seed=s)),
        ("CEM (matched, ~1000)", lambda m, s: RefCEM(m, num_samples=64, n_steps=16, topk=8, seed=s)),
        ("GD single-start (no clip)", lambda m, s: RefGD(m, n_steps=30, num_samples=1, lr=0.15, grad_clip=None, seed=s)),
        ("GD SGD hi-lr (diverges)", lambda m, s: RefGD_SGD(m, n_steps=30, num_samples=1, lr=20.0, seed=s)),
        ("GRASP (~1000)", lambda m, s: GRASPSolver(m, num_restarts=32, rounds=3, inner_steps=10,
                                                    elite=8, lr=0.15, grad_clip=1.0, seed=s)),
    ]

    rows = []
    for name, factory in planners:
        best, succ, tsum, ev, div = [], [], [], [], 0
        for s in SEEDS:
            g = torch.Generator().manual_seed(100 + s)
            targets = (torch.rand(N_ENVS, H, A, generator=g) * 4 - 2)   # in [-2,2]
            model = RastriginCostModel(targets)
            info = make_infos(targets, N_ENVS)
            info["_tgt"] = targets   # solver will expand to (bs,S,H,A)
            planner = factory(model, s)
            planner.configure(action_space=action_space, n_envs=N_ENVS, config=cfg)
            costs, dt, nev, diverged = run(planner, model, info)
            sane = torch.isfinite(costs) & (costs < 1e6)   # "blowup" counts as divergence
            best.append(costs[sane].mean().item() if sane.any() else float("nan"))
            succ.append((costs < SUCCESS_THRESH).float().mean().item())
            tsum.append(dt); ev.append(nev)
            div += int(diverged or (~sane).float().mean().item() > 0.5)
        rows.append((name, np.nanmean(best), np.mean(succ) * 100, np.mean(tsum),
                     int(np.mean(ev)), div))

    def fmt_cost(c):
        return "blowup" if (not np.isfinite(c)) else f"{c:.3f}"

    hdr = f"{'planner':<28}{'cost':>9}{'success%':>10}{'time(s)':>10}{'evals':>9}{'diverged':>10}"
    print("\n" + hdr); print("-" * len(hdr))
    for name, c, sr, t, e, d in rows:
        print(f"{name:<28}{fmt_cost(c):>9}{sr:>10.1f}{t:>10.3f}{e:>9d}{d:>8d}/3")
    print(f"\n64 problems x 3 seeds; success = global basin (cost < {SUCCESS_THRESH}).")
    print("Lower cost / higher success / lower time better. `evals` = cost calls "
          "(compute budget). `diverged` = seeds with non-finite cost.")

    with open("synthetic_results.md", "w") as f:
        f.write("| planner | mean cost | success % | wall-clock (s) | cost-evals | diverged |\n")
        f.write("|---|---:|---:|---:|---:|---:|\n")
        for name, c, sr, t, e, d in rows:
            f.write(f"| {name} | {fmt_cost(c)} | {sr:.1f} | {t:.3f} | {e} | {d}/3 |\n")
    print("wrote synthetic_results.md")


if __name__ == "__main__":
    main()


## 5. Sanity check: optimizer-level comparison (fast, CPU)

Stresses the three optimizers on a rugged differentiable cost (a proxy for
the world-model landscape) with matched compute budgets. Confirms GRASP
beats budget-matched CEM and all GD variants, and is stable.

In [ ]:
!python synthetic_benchmark.py

## 6. Push-T benchmark: success rate + wall-clock planning time

Closed-loop MPC in the real env, same start/goal pairs for every planner.

In [ ]:
%%bash
cd /content/le-wm; export STABLEWM_HOME=/content/.stable-wm
xvfb-run -a python benchmark.py \
  --model pusht/lewm/weights_epoch_20.pt --img-size 112 --num-eval 20 \
  --planners cem,gd,grasp

In [ ]:
print(open('/content/le-wm/grasp_benchmark.md').read())

## 7. Download results

In [ ]:
from google.colab import files
import shutil, os
os.makedirs('/content/w6_out', exist_ok=True)
for f in ['grasp_benchmark.md','synthetic_results.md','grasp_solver.py','benchmark.py']:
    p='/content/le-wm/'+f
    if os.path.exists(p): shutil.copy(p,'/content/w6_out/')
shutil.make_archive('/content/w6_out','zip','/content/w6_out')
files.download('/content/w6_out.zip')

## Notes

- **GD budgets**: `gd` is literal vanilla single-start Adam (clipped); add
  `gd_multi` to `--planners` for a stronger 32-start GD reference.
- Wall-clock is measured around each `solver.solve()` with CUDA sync, so it
  reflects real planning latency, not sampling budget.
- Raise `--num-eval` for tighter success-rate estimates (slower).